<a href="https://colab.research.google.com/github/cpython-projects/python_da_06_11_25/blob/main/lesson_29.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [40]:
!pip install requests
!pip install psycopg2
!pip install plotly
!pip install pandas
!pip install sqlalchemy

In [41]:
import pandas as pd
from sqlalchemy import create_engine, text
import plotly.express as px

In [42]:
DB_USER="da_last_lesson_user"
DB_PASS="UtV3qUNG6gYiW8ReXQvkeWY5nc7hldx2"
DB_HOST="dpg-d6nem4k50q8c739qvte0-a.oregon-postgres.render.com"
DB_PORT="5432"
DB_NAME="da_last_lesson"

In [43]:
engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

In [44]:
query = """
SELECT *
FROM portfolio
"""

df = pd.read_sql(text(query), engine)

In [45]:
df

,id,crypto_id,amount,buy_price,buy_date,sell_price,sell_date
0,5,bitcoin,0.10,63000.0,2026-01-10,NaN,None
1,6,ethereum,2.00,2500.0,2026-02-05,NaN,None
2,7,cardano,500.00,0.3,2026-02-20,NaN,None
3,8,bitcoin,0.05,64000.0,2026-01-15,60000.0,2026-02-15


In [63]:
query = """
SELECT *
FROM portfolio_history
"""

df = pd.read_sql(text(query), engine)
df

,id,date_time,total_value,total_profit
0,1,2026-03-09,10995.7615,-654.2385
1,2,2026-03-09,10987.8925,-662.1075
2,3,2026-03-10,11155.1825,-3494.8175


In [47]:
import requests

def get_current_prices(crypto_ids: list):
    ids_str = ','.join(crypto_ids)
    url = f"https://api.coingecko.com/api/v3/simple/price?ids={ids_str}&vs_currencies=usd"
    data = requests.get(url).json()
    return {key: value['usd'] for key, value in data.items()}

In [48]:
query = """
  SELECT DISTINCT crypto_id
  FROM portfolio
"""

df = pd.read_sql(text(query), engine)
type(df.crypto_id)

pandas.core.series.Series

In [49]:
for item in df.crypto_id:
  print(item)

cardano
bitcoin
ethereum


In [50]:
current_prices = get_current_prices(df.crypto_id)

In [51]:
query_portfolio = """
  SELECT * FROM portfolio;
"""

df_portfolio = pd.read_sql(query_portfolio, engine)
df_portfolio

,id,crypto_id,amount,buy_price,buy_date,sell_price,sell_date
0,5,bitcoin,0.10,63000.0,2026-01-10,NaN,None
1,6,ethereum,2.00,2500.0,2026-02-05,NaN,None
2,7,cardano,500.00,0.3,2026-02-20,NaN,None
3,8,bitcoin,0.05,64000.0,2026-01-15,60000.0,2026-02-15


In [52]:
df_portfolio['profit'] = (df_portfolio.sell_price - df_portfolio.buy_price) * df_portfolio.amount
df_portfolio

,id,crypto_id,amount,buy_price,buy_date,sell_price,sell_date,profit
0,5,bitcoin,0.10,63000.0,2026-01-10,NaN,None,NaN
1,6,ethereum,2.00,2500.0,2026-02-05,NaN,None,NaN
2,7,cardano,500.00,0.3,2026-02-20,NaN,None,NaN
3,8,bitcoin,0.05,64000.0,2026-01-15,60000.0,2026-02-15,-200.0


In [54]:
from pandas.core.dtypes.missing import isna
def process_row(row):
  if pd.isna(row['profit']):
    return current_prices.get(row['crypto_id']) * row['amount']
  return row['profit']


df_portfolio['current_portfolio'] = df_portfolio.apply(process_row, axis=1)
df_portfolio

,id,crypto_id,amount,buy_price,buy_date,sell_price,sell_date,profit,current_portfolio
0,5,bitcoin,0.10,63000.0,2026-01-10,NaN,None,NaN,7081.3000
1,6,ethereum,2.00,2500.0,2026-02-05,NaN,None,NaN,4139.6800
2,7,cardano,500.00,0.3,2026-02-20,NaN,None,NaN,134.2025
3,8,bitcoin,0.05,64000.0,2026-01-15,60000.0,2026-02-15,-200.0,-200.0000


In [53]:
expenses = (df_portfolio.amount * df_portfolio.buy_price).sum()
expenses

np.float64(14650.0)

In [56]:
current_portfolio = df_portfolio.current_portfolio.sum()
current_portfolio

np.float64(11155.182499999999)

In [57]:
profit = current_portfolio - expenses
profit

np.float64(-3494.817500000001)

In [59]:
import datetime

query = text("INSERT INTO portfolio_history (date_time, total_value, total_profit) VALUES (:d, :v, :t)")

In [62]:
with engine.begin() as conn:
  conn.execute(
      query,
      {'d': datetime.datetime.now(), 'v': float(current_portfolio), 't': float(profit)}
  )